# Task 049: structured-light-python-main — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: 3D structured light reconstruction using phase-shifting profilometry

**Paper**: See repository documentation
**Repository**: https://github.com/elerac/structuredlight

### Physical Background

Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.

### Forward Model

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

### Inverse Problem

```
Nonlinear least-squares fitting or matrix decomposition (MCR-ALS, NMF)
```

### Performance Metrics
- **PSNR**: 146.80 dB
- **SSIM**: N/A


### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
import sys
import os
import math
import numpy as np
import cv2
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Any, Dict

# ==========================================
# 1. LOAD AND PREPROCESS DATA
# ==========================================

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`load_and_preprocess_data`

In [ ]:
# ==========================================
# 1. LOAD AND PREPROCESS DATA
# ==========================================
def load_and_preprocess_data(
    width: int,
    height: int,
    frequencies: List[float],
    shift_num: int
) -> Dict[str, Any]:
    """
    Generates simulation configuration and phase-shifting patterns.
    Acts as the data loader for this simulation-based problem.
    """
    
    # Helper to create patterns
    def create_psp_patterns(w, h, freqs, shifts, vertical=True):
        patterns = []
        # Standard N-step phase shifting angles
        phase_shifts_angles = [2 * np.pi / shifts * i for i in range(shifts)]
        
        if vertical:
            length = w
        else:
            length = h
            
        x = np.linspace(0, length, length)
        
        freq_patterns = []
        for frequency in freqs:
            step_patterns = []
            for phase_shift in phase_shifts_angles:
                # Cosine pattern generation: 0.5 + 0.5 * cos(...)
                cos_val = 0.5 + 0.5 * np.cos(2 * np.pi * frequency * (x / length) - phase_shift)
                if vertical:
                    pat = np.tile(cos_val, (h, 1))
                else:
                    pat = np.tile(cos_val.reshape((-1, 1)), (1, w))
                step_patterns.append(pat)
            freq_patterns.append(step_patterns)
        return freq_patterns, phase_shifts_angles

    # Generate patterns
    patterns_v, _ = create_psp_patterns(width, height, frequencies, shift_num, vertical=True)
    patterns_h, phase_shifts = create_psp_patterns(width, height, frequencies, shift_num, vertical=False)

    # Define calibration data (Hardcoded for simulation context)
    f = 1000.0
    cx = width / 2.0
    cy = height / 2.0
    mtx = np.array([[f, 0, cx], [0, f, cy], [0, 0, 1]])
    dist = np.zeros(5)
    R = np.eye(3)
    # Baseline T = [100, 0, 0], Camera 2 is shifted by 100 units relative to Camera 1
    T = np.array([[100.0], [0.0], [0.0]]) 

    calibration_data = {
        'camera_0': {'mtx': mtx, 'dist': dist},
        'camera_1': {'mtx': mtx, 'dist': dist},
        'R': R,
        'T': T
    }

    data = {
        'frequencies': frequencies,
        'phase_shifts': phase_shifts,
        'patterns_vertical': patterns_v,
        'patterns_horizontal': patterns_h,
        'calibration': calibration_data,
        'image_shape': (height, width)
    }
    
    return data

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

Functions: `forward_operator`

In [ ]:
# ==========================================
# 2. FORWARD OPERATOR
# ==========================================
def forward_operator(
    patterns_dict: Dict[str, List[List[np.ndarray]]], 
    disparity_pixel_shift: int = 20
) -> Dict[str, List[List[List[np.ndarray]]]]:
    """
    Simulates the projection and capture process.
    Input: Dictionary of projection patterns (vertical/horizontal).
    Output: Captured images for Camera 0 and Camera 1.
    Physics: Cam 0 sees pattern exactly. Cam 1 sees it shifted (disparity).
    """
    
    captured_data = {
        'cam1_vertical': [], 'cam2_vertical': [],
        'cam1_horizontal': [], 'cam2_horizontal': []
    }
    
    # Process Vertical Fringes
    # Structure: patterns_dict['vertical'][freq_idx][shift_idx]
    for freq_idx, pattern_group in enumerate(patterns_dict['patterns_vertical']):
        c1_group = []
        c2_group = []
        for pattern in pattern_group:
            # Camera 0: Identity (Projector view in this simplified model)
            img1 = pattern.copy()
            
            # Camera 1: Simulated Disparity Shift
            img2 = np.roll(img1, disparity_pixel_shift, axis=1)
            # Handle edge artifacts from roll
            if disparity_pixel_shift > 0:
                img2[:, :disparity_pixel_shift] = 0
            elif disparity_pixel_shift < 0:
                img2[:, disparity_pixel_shift:] = 0
                
            c1_group.append(img1)
            c2_group.append(img2)
        captured_data['cam1_vertical'].append(c1_group)
        captured_data['cam2_vertical'].append(c2_group)

    # Process Horizontal Fringes
    for freq_idx, pattern_group in enumerate(patterns_dict['patterns_horizontal']):
        c1_group = []
        c2_group = []
        for pattern in pattern_group:
            img1 = pattern.copy()
            
            # Camera 1: Same horizontal shift (epipolar geometry assumption)
            img2 = np.roll(img1, disparity_pixel_shift, axis=1)
            if disparity_pixel_shift > 0:
                img2[:, :disparity_pixel_shift] = 0
            elif disparity_pixel_shift < 0:
                img2[:, disparity_pixel_shift:] = 0
                
            c1_group.append(img1)
            c2_group.append(img2)
        captured_data['cam1_horizontal'].append(c1_group)
        captured_data['cam2_horizontal'].append(c2_group)
        
    return captured_data

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Nonlinear least-squares fitting or matrix decomposition (MCR-ALS, NMF)
```

Functions: `run_inversion`

In [ ]:
# ==========================================
# 3. RUN INVERSION
# ==========================================
def run_inversion(
    captured_data: Dict[str, Any],
    frequencies: List[float],
    phase_shifts: List[float],
    calibration: Dict[str, Any]
) -> np.ndarray:
    """
    Performs Phase Wrapping, Unwrapping, Phasogrammetry (Matching), and Triangulation.
    """

    # --- 3a. Phase Calculation Helper ---
    def calculate_phase(images_freq_group, shifts):
        # images_freq_group: List[np.ndarray] (N steps)
        imgs = np.array(images_freq_group)
        shifts_arr = np.array(shifts).reshape((-1, 1, 1))
        
        # Least squares phase extraction (or standard N-step formula)
        # formula: tan(phi) = - sum(I * sin(delta)) / sum(I * cos(delta)) 
        # Note: The input code used a sum based approach equivalent to DFT 
        
        sin_part = np.sum(imgs * np.sin(shifts_arr), axis=0)
        cos_part = np.sum(imgs * np.cos(shifts_arr), axis=0)
        
        wrapped_phase = np.arctan2(sin_part, cos_part)
        magnitude = 2 * np.sqrt(sin_part**2 + cos_part**2) / len(images_freq_group)
        
        return wrapped_phase, magnitude

    # --- 3b. Temporal Phase Unwrapping Helper ---
    def unwrap_phases(wrapped_phases_list, freqs):
        # Multi-frequency temporal unwrapping
        # Start with lowest frequency (usually freq=1, where phase range is -pi to pi over whole image)
        
        unwrapped = wrapped_phases_list[0] # Base phase
        
        for i in range(1, len(freqs)):
            p_low = unwrapped
            p_high = wrapped_phases_list[i]
            lambda_l = 1.0 / freqs[i-1]
            lambda_h = 1.0 / freqs[i]
            
            # k = round( ( (lambda_l/lambda_h) * p_low - p_high ) / 2pi )
            ratio = lambda_l / lambda_h
            k = np.round((ratio * p_low - p_high) / (2 * np.pi))
            unwrapped = p_high + 2 * np.pi * k
            
        return unwrapped

    # --- 3c. Process All Cameras/Orientations ---
    # We need unwrapped phase maps for: 
    # Cam1 Vertical (p1_v), Cam2 Vertical (p2_v) -> X coordinate encoding
    # Cam1 Horizontal (p1_h), Cam2 Horizontal (p2_h) -> Y coordinate encoding
    
    phase_maps = {}
    keys_map = [
        ('cam1_vertical', 'p1_v'), ('cam2_vertical', 'p2_v'),
        ('cam1_horizontal', 'p1_h'), ('cam2_horizontal', 'p2_h')
    ]
    
    for input_key, output_key in keys_map:
        wrapped_list = []
        for f_idx, imgs in enumerate(captured_data[input_key]):
            w_phase, _ = calculate_phase(imgs, phase_shifts)
            wrapped_list.append(w_phase)
        
        unwrapped = unwrap_phases(wrapped_list, frequencies)
        phase_maps[output_key] = unwrapped

    # --- 3d. Phasogrammetry (Matching) ---
    p1_h = phase_maps['p1_h'] # Y-encoding Cam 1
    p1_v = phase_maps['p1_v'] # X-encoding Cam 1
    p2_h = phase_maps['p2_h'] # Y-encoding Cam 2
    p2_v = phase_maps['p2_v'] # X-encoding Cam 2
    
    h, w = p1_h.shape
    step = 50 # Subsampling for speed
    
    # Define ROI (crop borders)
    roi_margin = 100
    xx = np.arange(roi_margin, w - roi_margin, step)
    yy = np.arange(roi_margin, h - roi_margin, step)
    
    pts1_list = []
    pts2_list = []

    # Optimization: vectorize finding corresponding points?
    # The legacy code iterates. We must implement the logic.
    # To find match for (x, y) in Cam1:
    # 1. Get phase values phi_x, phi_y at (x,y) in Cam1.
    # 2. Find (u, v) in Cam2 such that P2_v(u,v) approx phi_x AND P2_h(u,v) approx phi_y.
    
    # Pre-computation for matching speedup isn't strictly forbidden but let's stick to the logic provided.
    # However, iterating is slow in Python. Let's do a constrained search based on epipolar geometry 
    # implicitly provided by the simulation (horizontal shift only).
    
    for y in yy:
        for x in xx:
            target_ph_h = p1_h[y, x]
            target_ph_v = p1_v[y, x]
            
            # In rectified stereo (or this simulation), corresponding point is on the same scanline y
            # So we only search along row y in Cam2 (or close to it)
            # Scan a window around y
            y_search_start = max(0, y - 2)
            y_search_end = min(h, y + 3)
            
            # Extract strips
            strip_p2_h = p2_h[y_search_start:y_search_end, :]
            strip_p2_v = p2_v[y_search_start:y_search_end, :]
            
            # Find candidate columns where phase matches
            # Metric: Euclidean distance in phase space
            dist_map = np.sqrt((strip_p2_h - target_ph_h)**2 + (strip_p2_v - target_ph_v)**2)
            
            min_val = np.min(dist_map)
            if min_val < 0.2: # Phase matching threshold
                # Get local coordinates of minimum
                loc_y, loc_x = np.unravel_index(np.argmin(dist_map), dist_map.shape)
                match_x = loc_x
                match_y = y_search_start + loc_y
                
                pts1_list.append([x, y])
                pts2_list.append([match_x, match_y])
                
    pts1 = np.array(pts1_list, dtype=np.float32)
    pts2 = np.array(pts2_list, dtype=np.float32)

    if len(pts1) == 0:
        return np.zeros((0, 3))

    # --- 3e. Triangulation ---
    cam1_mtx = calibration['camera_0']['mtx']
    cam2_mtx = calibration['camera_1']['mtx']
    dist1 = calibration['camera_0']['dist']
    dist2 = calibration['camera_1']['dist']
    R_mat = calibration['R']
    T_vec = calibration['T']
    
    # Projection matrices
    # P1 = K1 * [I | 0]
    P1 = np.dot(cam1_mtx, np.hstack((np.eye(3), np.zeros((3, 1)))))
    # P2 = K2 * [R | T]
    P2 = np.dot(cam2_mtx, np.hstack((R_mat, T_vec)))
    
    # Undistort points
    # Reshape to (N, 1, 2) for cv2
    pts1_cv = np.ascontiguousarray(pts1[:, :2]).reshape((pts1.shape[0], 1, 2))
    pts2_cv = np.ascontiguousarray(pts2[:, :2]).reshape((pts2.shape[0], 1, 2))
    
    pts1_undist = cv2.undistortPoints(pts1_cv, cam1_mtx, dist1, P=cam1_mtx)
    pts2_undist = cv2.undistortPoints(pts2_cv, cam2_mtx, dist2, P=cam2_mtx)
    
    # Triangulate
    points_4d_hom = cv2.triangulatePoints(P1, P2, pts1_undist, pts2_undist)
    
    # Convert from homogeneous
    points_3d = cv2.convertPointsFromHomogeneous(points_4d_hom.T)
    points_3d = points_3d.reshape(-1, 3)
    
    return points_3d

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `evaluate_results`

In [ ]:
# ==========================================
# 4. EVALUATE RESULTS
# ==========================================
def evaluate_results(points_3d: np.ndarray, expected_depth: float = 5000.0) -> None:
    """
    Calculates statistics on reconstructed point cloud and compares to theoretical model.
    """
    if points_3d.shape[0] == 0:
        print("EVALUATION FAILED: No 3D points reconstructed.")
        return

    z_vals = points_3d[:, 2]
    z_mean = np.mean(z_vals)
    z_std = np.std(z_vals)
    
    error = abs(z_mean - expected_depth)
    
    print("\n" + "="*30)
    print("EVALUATION REPORT")
    print("="*30)
    print(f"Reconstructed Points: {points_3d.shape[0]}")
    print(f"Mean Z Depth:         {z_mean:.4f} mm")
    print(f"Std Dev Z:            {z_std:.4f} mm")
    print(f"Theoretical Z:        {expected_depth:.4f} mm")
    print(f"Absolute Error:       {error:.4f} mm")
    
    # Tolerance check
    if error < 50.0: # Generous tolerance for discrete simulation
        print("STATUS: SUCCESS (Within tolerance)")
    else:
        print("STATUS: WARNING (High deviation)")
        
    # Calculate simple PSNR proxy (Linearity check logic from original code)
    # Since we don't have the phase map here, we check planarity of Z
    # Ideal surface is flat (Z = constant)
    mse = np.mean((z_vals - z_mean)**2)
    if mse == 0:
        psnr = 100
    else:
        max_val = np.max(z_vals)
        psnr = 20 * math.log10(max_val / math.sqrt(mse))
    
    print(f"Surface Planarity PSNR: {psnr:.2f} dB")
    print("="*30 + "\n")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# Configuration
PROJ_W, PROJ_H = 1280, 720
FREQUENCIES = [1, 4, 12]
SHIFTS_NUM = 4
PIXEL_DISPARITY = 20

### 1. Load Data

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 1. Load Data
print("[1/4] Loading and Preprocessing Data...")
sim_data = load_and_preprocess_data(PROJ_W, PROJ_H, FREQUENCIES, SHIFTS_NUM)

### 2. Forward Model (Simulation)

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 2. Forward Model (Simulation)
print("[2/4] Running Forward Operator (Simulation)...")
captured_images = forward_operator(
    sim_data, 
    disparity_pixel_shift=PIXEL_DISPARITY
)

### 3. Inverse Problem

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 3. Inverse Problem
print("[3/4] Running Inversion (Phase Unwrapping & Triangulation)...")
reconstructed_points = run_inversion(
    captured_images,
    sim_data['frequencies'],
    sim_data['phase_shifts'],
    sim_data['calibration']
)

### 4. Evaluation

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 4. Evaluation
print("[4/4] Evaluating Results...")
# Theoretical depth Z = f * Baseline / Disparity
# f=1000, Baseline=100, Disparity=20
THEORETICAL_Z = (1000.0 * 100.0) / float(PIXEL_DISPARITY)

evaluate_results(reconstructed_points, expected_depth=THEORETICAL_Z)

print("OPTIMIZATION_FINISHED_SUCCESSFULLY")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **structured-light-python-main**:

1. **Problem Setup**: Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=146.80 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: See documentation
- Repository: https://github.com/elerac/structuredlight
